# Notebook 1 – Laser scans and the lidar position

In this notebook you look **inside a laser scan** and see why the lidar
position in `config/robot.yaml` matters.

You will:
1. get one `sensor_msgs/msg/LaserScan` message and look at its fields,
2. turn the distances into points (polar → Cartesian),
3. move the points from the lidar frame `laser` into the robot frame `base_link`,
4. see what wrong values in `robot.yaml` do – including the "turning on the spot" test.

Tutorial: [chapter 2 – message types](../../TUTORIAL_SLAM_NAVIGATION.md#the-three-most-important-message-types),
[section 6.4 – where is the lidar on the robot?](../../TUTORIAL_SLAM_NAVIGATION.md#64-where-is-the-lidar-on-the-robot)

**How to use this notebook**

- Run the cells from top to bottom with **Shift + Enter**.
- `LIVE = False` uses the sample data in `sample_data/` – no robot needed.
- `LIVE = True` takes the data from the running robot. Start Jupyter from a
  terminal in which ROS 2 and the workspace are sourced (see `README.md` in this folder).
- Change values, run the cells again, look at what changes. You can't break anything.

**Live mode needs:** `robot.launch.py` (or `mapping.launch.py`) running on the Jetson.

In [ ]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
from nb_helpers import *

LIVE = False     # True: read /scan from the running robot

## 1. Get one scan

In live mode we subscribe to `/scan`, wait for **one** message and unsubscribe
again – exactly what `ros2 topic echo /scan --once` does. The QoS profile
`qos_profile_sensor_data` ("best effort") is the usual choice for sensor data.

In [ ]:
if LIVE:
    from rclpy.qos import qos_profile_sensor_data
    from sensor_msgs.msg import LaserScan
    msg = receive_one('/scan', LaserScan, qos_profile_sensor_data)
    scan = scan_from_msg(msg)
else:
    scan = load_scan(SAMPLE_SCAN)

r = scan['ranges']
print('frame_id        :', scan['frame_id'])
print('number of beams :', len(r))
print(f"angle_min       : {scan['angle_min']:.3f} rad = {math.degrees(scan['angle_min']):.1f} deg")
print(f"angle_increment : {scan['angle_increment']:.5f} rad = {math.degrees(scan['angle_increment']):.2f} deg")
print(f"valid range     : {scan['range_min']} ... {scan['range_max']} m")
print('invalid readings:', int(np.sum(~np.isfinite(r))), '(inf = nothing hit or no echo)')

Want to keep a real scan for later (e.g. to work at home)? Uncomment and run:

In [ ]:
# save_scan(scan, 'my_scan.yaml')          # later: scan = load_scan('my_scan.yaml')

## 2. What is inside?

A scan does **not** contain points. It contains a list of **distances**
(`ranges`). The angle of reading number `i` is not stored – you compute it:

$$\text{angle}_i = \text{angle\_min} + i \cdot \text{angle\_increment}$$

Angle 0 is the lidar's own "forward" direction. Angles grow counter-clockwise.

In [ ]:
angles = scan_angles(scan)
for i in range(0, len(r), len(r) // 12):
    print(f'beam {i:4d}:  angle {math.degrees(angles[i]):7.1f} deg  ->  range {r[i]:.3f} m')

ahead = np.argmin(np.abs(angles))          # the beam closest to angle 0
print(f'\nStraight ahead of the LIDAR (beam {ahead}): {r[ahead]:.3f} m')

**Question:** In the sample scan, the robot stands at x = 1.0 m, y = 0.2 m and
looks along +x. Which wall does the forward beam hit? (Look at the map in
notebook 2: there is a wall at x = 3.0 m.)

## 3. From distances to points

Each reading is a point in **polar coordinates** (distance, angle). For
plotting and for TF we need **Cartesian coordinates**:

$$x = r \cos(\text{angle}), \qquad y = r \sin(\text{angle})$$

`scan_to_points()` does this and drops invalid readings (`inf`, too close, too far).

In [ ]:
x, y = scan_to_points(scan)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(x, y, 'r.', markersize=3, label='scan points')
ax.plot(0, 0, 'k+', markersize=15, mew=2, label='lidar')
ax.arrow(0, 0, 0.5, 0, width=0.03, color='k')
ax.text(0.55, 0.05, '0 deg of the lidar')
ax.set_aspect('equal'); ax.grid(True); ax.legend(loc='lower left')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title("Scan in the lidar's own frame: 'laser'")
plt.show()

## 4. Into the robot frame `base_link`

The points above are relative to the **lidar**. Nav2 and SLAM need them
relative to the **robot center** (`base_link`). This is what the static
transform `base_link → laser` from `robot.yaml` does:

$$\begin{pmatrix} x_b \\ y_b \end{pmatrix} =
\begin{pmatrix} x_{lidar} \\ y_{lidar} \end{pmatrix} +
\begin{pmatrix} \cos\psi & -\sin\psi \\ \sin\psi & \cos\psi \end{pmatrix}
\begin{pmatrix} x \\ y \end{pmatrix}$$

with the lidar position $(x_{lidar}, y_{lidar})$ = (`x_cm`, `y_cm`) and the
rotation $\psi$ = `yaw_deg`. TF does exactly this for every point.

In [ ]:
config = load_robot_config()
lidar = config['lidar']
print('lidar section of robot.yaml:', lidar)

lx, ly, lyaw = lidar['x_cm'] / 100, lidar['y_cm'] / 100, math.radians(lidar['yaw_deg'])
bx, by = transform_points(x, y, lx, ly, lyaw)

radius = config['robot']['radius_cm'] / 100
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(bx, by, 'r.', markersize=3, label='scan points')
draw_robot(ax, 0, 0, 0, radius, label='robot (base_link)')
ax.plot(lx, ly, 'ks', label='lidar position')
ax.set_aspect('equal'); ax.grid(True); ax.legend(loc='lower left')
ax.set_xlabel('x [m] (forward)'); ax.set_ylabel('y [m] (left)')
ax.set_title("Scan in the robot frame: 'base_link'")
plt.show()

**Check 2 from the tutorial (distance):** how far is the nearest obstacle
straight in front of the **bumper**? With the real robot, compare this with a
measuring tape.

In [ ]:
front = (bx > 0) & (np.abs(by) < 0.05)        # points in a 10 cm wide stripe ahead
center_to_obstacle = bx[front].min()
print(f'robot center -> obstacle ahead: {center_to_obstacle:.3f} m')
print(f'bumper       -> obstacle ahead: {center_to_obstacle - radius:.3f} m')

## 5. Experiment: the wrong `yaw_deg`

What happens if the lidar is mounted turned around, but `robot.yaml` says
`yaw_deg: 0`? Below we convert the same scan once with the correct and once
with a wrong rotation.

In [ ]:
WRONG_YAW_DEG = lidar['yaw_deg'] + 180       # try 90, 10, -90 ...

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, yaw_deg, title in [(axes[0], lidar['yaw_deg'], 'correct yaw'),
                           (axes[1], WRONG_YAW_DEG, f'wrong yaw ({WRONG_YAW_DEG} deg)')]:
    px, py = transform_points(x, y, lx, ly, math.radians(yaw_deg))
    ax.plot(px, py, 'r.', markersize=3)
    draw_robot(ax, 0, 0, 0, radius)
    ax.set_aspect('equal'); ax.grid(True); ax.set_title(title)
    ax.set_xlabel('x [m] (forward)'); ax.set_ylabel('y [m] (left)')
plt.show()

With 180° wrong, everything that is **in front** of the robot appears
**behind** it. Nav2 would drive straight into the wall – that's why Check 1 in
the tutorial must be done **before** mapping.

## 6. Experiment: the wrong `x_cm` – turning on the spot

This is Check 3 from the tutorial, simulated. We use the sample map as "the
world":

1. The lidar is **really** mounted `TRUE_X_CM` in front of the robot center.
2. The robot turns on the spot (8 headings, every 45°). At each heading we
   simulate a scan from the **real** lidar position.
3. We convert every scan into the map with the lidar position we **assume**
   (= what is written in `robot.yaml`) and draw all 8 scans on top of each other.

If the assumption is right, all scans land on the same walls. If not, every
scan is shifted in a different direction – the walls get blurred.

In [ ]:
grid = GridMap(SAMPLE_MAP)
TRUE_X_CM, TRUE_Y_CM = 10.0, 0.0          # where the lidar REALLY is
ROBOT_X, ROBOT_Y = 1.0, 0.2               # the robot turns on the spot here
beam_angles = np.radians(np.arange(-180, 180, 1.0))


def turning_on_the_spot(assumed_x_cm, assumed_y_cm):
    """Simulate 8 scans while turning; convert them with the ASSUMED lidar position."""
    xs, ys = [], []
    for heading in np.radians(np.arange(0, 360, 45)):
        # where the real lidar is in the map at this heading
        real_lx, real_ly = transform_points(TRUE_X_CM / 100, TRUE_Y_CM / 100, ROBOT_X, ROBOT_Y, heading)
        ranges = grid.raycast(real_lx, real_ly, heading + beam_angles)
        ok = np.isfinite(ranges)
        px, py = ranges[ok] * np.cos(beam_angles[ok]), ranges[ok] * np.sin(beam_angles[ok])  # laser
        px, py = transform_points(px, py, assumed_x_cm / 100, assumed_y_cm / 100, 0.0)      # base_link
        px, py = transform_points(px, py, ROBOT_X, ROBOT_Y, heading)                          # map
        xs.append(px); ys.append(py)
    return np.concatenate(xs), np.concatenate(ys)


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, assumed in [(axes[0], TRUE_X_CM), (axes[1], 0.0)]:
    show_map(ax, grid, f'real lidar x = {TRUE_X_CM:.0f} cm, robot.yaml says x_cm = {assumed:.0f}')
    px, py = turning_on_the_spot(assumed, TRUE_Y_CM)
    ax.plot(px, py, 'r.', markersize=1)
plt.show()

Left: sharp walls. Right: every wall is doubled or blurred by about
2 × 10 cm – this is what you see in RViz (Fixed Frame `odom`, Decay Time 10)
when `x_cm` is wrong. SLAM would draw a blurred map from such scans.

## Exercises

1. Change `TRUE_X_CM` / the assumed value in the last cell: how big must the
   error be until you clearly see it? Try 2 cm, 5 cm, 20 cm.
2. Set `TRUE_Y_CM = -8` and keep the assumed `y` at 0. What does the blur look like now?
3. Live: put a box exactly 50 cm in front of the bumper (measuring tape) and
   run sections 1–4 with `LIVE = True`. Does "bumper → obstacle ahead" say 0.50 m?
   If not, which value in `robot.yaml` is wrong?
4. Live: put an object to the **left** of the robot. Does it appear at positive `y`
   in the `base_link` plot?